In [1]:
import os
import time 
import re
import requests
import polars as pl

from decorator import append
from dotenv import load_dotenv
from duckdb.experimental.spark.sql.functions import length
from datetime import datetime

load_dotenv(override=True)

RIOT_PERSONAL_API_KEY = os.getenv("LOL_PERSONAL_API_KEY")
POSTGRES_LOCAL_URL = os.getenv("POSTGRES_LOCAL_URL")

In [2]:
_camel_case_regex = re.compile(
    r"(?<=[a-z])(?=[A-Z])"  # camelCase: survive|Min
    r"|(?<=[A-Z])(?=[A-Z][a-z])"  # acronym: HTML|Parser
    r"|(?<=[a-zA-Z])(?=\d)"  # letter→digit: survive|15
    r"|(?<=\d)(?=[a-zA-Z])"  # digit→letter: 15|min
)


def camel_to_snake(input_str: str) -> str:
    return _camel_case_regex.sub("_", input_str).lower()


def snake_all_keys(json_obj: dict):
    if isinstance(json_obj, dict):
        return {camel_to_snake(key): snake_all_keys(value) for key, value in json_obj.items()}

    if isinstance(json_obj, list):
        return [snake_all_keys(value) for value in json_obj]

    return json_obj


In [3]:
players = pl.read_database_uri("SELECT * FROM players", POSTGRES_LOCAL_URL)
players

puuid,game_name,tag_line,region,synced_at
str,str,str,str,datetime[μs]
"""x1jqYLksFHSjj-V_dTeZrzR9fFEVDt…","""bird raven""","""0000""","""EUN1""",2026-05-31 18:10:37.551072
"""QMPPN4oUrY-FW-BBOZ34yDZZmjghPa…","""KaizokuNoOni""","""2829""","""EUN1""",2026-05-31 18:10:37.860377
"""yf1OVyzGtKFWKAh__NIUFtnEisdohM…","""Nolba""","""EUNE""","""EUN1""",2026-05-31 18:10:38.176115
"""gw0udTm33m9cDPfENXndpjlXmkHwxl…","""Defenser""","""EUNE""","""EUN1""",2026-05-31 18:10:38.472933
"""tD8QYcsn9jmPaM7AACN9Id2fwgusfN…","""Dioklis""","""EUNE""","""EUN1""",2026-05-31 18:10:38.757716
…,…,…,…,…
"""dYiEC-ravKg74XNQMp4KsOp4S7aVlR…","""black nails""","""988""","""EUN1""",2026-05-31 19:15:39.289847
"""ZLw_1J27yiyrFC_UN7-Uck-IyYMBLL…","""AleEmotkaZostała""","""RMJ""","""EUN1""",2026-05-31 19:15:39.630180
"""0CPr6RzHVit2UWf0Cu0EXPbIj77eko…","""chiefelo""","""0314""","""EUN1""",2026-05-31 19:15:39.980448


In [19]:
from datetime import date


BASE_EUN1_URL = "https://eun1.api.riotgames.com"

headers = {
    'X-Riot-Token': RIOT_PERSONAL_API_KEY
}

def get_player_snapshot(puuid: str):
    player_icon_url = f"{BASE_EUN1_URL}/lol/summoner/v4/summoners/by-puuid/{puuid}"
    response_icon = requests.get(player_icon_url, headers=headers)

    if response_icon.status_code != 200:
        print("request failed with status code on getting player icon", response_icon.status_code)
        return None
    
    response_icon_json = response_icon.json()

    player_challanges_url =f"{BASE_EUN1_URL}/lol/challenges/v1/player-data/{puuid}"
    response_challanges = requests.get(player_challanges_url,headers=headers)

    if response_challanges.status_code != 200:
        print("request failed with status code on getting player challanges", response_challanges.status_code)
        return None
    
    response_challanges_json = response_challanges.json();
    
    total_points = {
        "level": response_challanges_json["totalPoints"]["level"],
        "current" : response_challanges_json["totalPoints"]["current"],
        "max":response_challanges_json["totalPoints"]["max"],
        "percentile":response_challanges_json["totalPoints"]["percentile"]
    }

    player_snapshot = {
        "puuid": puuid,
        "snapshot_date":date.today(),
        "summoner_level" : response_icon_json["summonerLevel"],
        "title":response_challanges_json["preferences"].get("title",None),
        "profile_icon_id":response_icon_json["profileIconId"],
        "crest_border":response_challanges_json["preferences"].get("crestBorder",None),
        "banner_accent":response_challanges_json["preferences"].get("bannerAccent",None),
        "prestige_crest_border_level":response_challanges_json["preferences"].get("prestigeCrestBorderLevel",None),
        "total_points" : total_points

    }

    return player_snapshot
    
get_player_snapshot(players["puuid"][0])

{'puuid': 'x1jqYLksFHSjj-V_dTeZrzR9fFEVDtcRFuZ0u9novIzvQW-QF-dE9YyTxsi08KU5rAUNccN9F2wLXQ',
 'snapshot_date': datetime.date(2026, 6, 1),
 'summoner_level': 158,
 'title': '10320704',
 'profile_icon_id': 3789,
 'crest_border': '1',
 'banner_accent': '',
 'prestige_crest_border_level': 150,
 'total_points': {'level': 'SILVER',
  'current': 3365,
  'max': 29680,
  'percentile': 0.335}}

In [5]:
player_snapshot_arr = []

for player in players.iter_rows(named=True):
    try:
        player_snapshot = get_player_snapshot(player["puuid"])
        if player_snapshot is None:
            time.sleep(31)
            continue
        player_snapshot_arr.append(player_snapshot)
    except Exception as e:
        print(f"Error fetching snapshot for {player['puuid']}: {e}")
        continue

player_snapshot_arr

request failed with status code on getting player icon 429
request failed with status code on getting player icon 429
request failed with status code on getting player icon 429
request failed with status code on getting player icon 429
request failed with status code on getting player icon 429
request failed with status code on getting player icon 429
request failed with status code on getting player icon 429
request failed with status code on getting player icon 429
request failed with status code on getting player icon 429
request failed with status code on getting player icon 429
request failed with status code on getting player icon 429
request failed with status code on getting player icon 429
request failed with status code on getting player icon 429
request failed with status code on getting player icon 429
request failed with status code on getting player icon 429
request failed with status code on getting player icon 429
request failed with status code on getting player icon 4

[{'summoner_level': 158,
  'title': '10320704',
  'profile_icon_id': 3789,
  'crest_border': '1',
  'banner_accent': '',
  'prestige_crest_border_level': 150,
  'total_points': {'level': 'SILVER',
   'current': 3365,
   'max': 29680,
   'percentile': 0.335}},
 {'summoner_level': 84,
  'title': '1420',
  'profile_icon_id': 7063,
  'crest_border': '1',
  'banner_accent': '2',
  'prestige_crest_border_level': 75,
  'total_points': {'level': 'GOLD',
   'current': 4595,
   'max': 29680,
   'percentile': 0.203}},
 {'summoner_level': 32,
  'title': '1',
  'profile_icon_id': 15,
  'crest_border': '1',
  'banner_accent': '1',
  'prestige_crest_border_level': 30,
  'total_points': {'level': 'BRONZE',
   'current': 1485,
   'max': 29680,
   'percentile': 0.434}},
 {'summoner_level': 108,
  'title': '40240802',
  'profile_icon_id': 657,
  'crest_border': '1',
  'banner_accent': '1',
  'prestige_crest_border_level': 75,
  'total_points': {'level': 'GOLD',
   'current': 4625,
   'max': 29680,
   'pe

In [6]:
player_snapshot_df=pl.DataFrame(player_snapshot_arr)
player_snapshot_df

summoner_level,title,profile_icon_id,crest_border,banner_accent,prestige_crest_border_level,total_points
i64,str,i64,str,str,i64,struct[4]
158,"""10320704""",3789,"""1""","""""",150,"{""SILVER"",3365,29680,0.335}"
84,"""1420""",7063,"""1""","""2""",75,"{""GOLD"",4595,29680,0.203}"
32,"""1""",15,"""1""","""1""",30,"{""BRONZE"",1485,29680,0.434}"
108,"""40240802""",657,"""1""","""1""",75,"{""GOLD"",4625,29680,0.203}"
143,"""40130204""",6893,"""1""","""""",100,"{""GOLD"",8455,29680,0.203}"
…,…,…,…,…,…,…
126,"""""",31,"""2""","""1""",75,"{""PLATINUM"",11030,29680,0.101}"
1100,"""""",6255,"""1""","""8""",0,"{""DIAMOND"",19015,29680,0.036}"
968,"""""",6486,"""2""","""15""",0,"{""DIAMOND"",20570,29680,0.036}"


In [16]:
import json

(player_snapshot_df
.with_columns(pl.col("total_points").map_elements(json.dumps).alias("total_points"))
.with_columns(pl.col(pl.Int64).cast(pl.Int32),pl.col(pl.Float64).cast(pl.Float32))
.write_database(
    table_name="player_snapshots",
    connection=POSTGRES_LOCAL_URL,
    if_table_exists="append",
    engine="adbc"
))

ProgrammingError: INVALID_ARGUMENT: [libpq] Failed to execute COPY statement: PGRES_FATAL_ERROR ERROR:  wrong number of columns: 2065853541, expected 4
CONTEXT:  COPY player_snapshots, line 1, column total_points
. SQLSTATE: 42804

In [17]:
player_snapshot_df.write_parquet(file="player_snapshot.parquet")